In [1]:
using Lux, LuxCUDA, CUDA, Optimisers, Zygote
using TensorBoardLogger, Logging
using Random
using Statistics
using JLD2
using MAT
using CairoMakie

include("lib/utils.jl")
using .Utils

# Models to try

- 8x8x128 pretty good
- 4x4x256 some degredation

# Load Data

Julia memmapped backing data.

In [2]:
function load()
    base = "build/train"
	open("$base/count.ser", "r") do io
		N = deserialize(io, Int64)
		Xs = Vector{AbstractArray{Float32, 4}}(undef, N)
		μs = Vector{AbstractArray{Float32, 3}}(undef, N)
		ys = Vector{Vector{Int}}(undef, N)

		yf = open("$base/ys.ser", "r")
		Xf = MMapReader("$base/Xs.ser")
		μf = open("$base/μs.ser", "r")
        count = 0
		for i ∈ 1:N
			Xs[i] = deserialize(Xf, Array{Float32})
			μs[i] = deserialize(μf, Array{Float32})
			ys[i] = deserialize(yf, Array{Int64})            
            count += length(ys[i])
		end
		close(yf)
		close(Xf)
		close(μf)
		return count, ys, Xs, μs
	end
end

@info "Loading"
@time N, ys, Xs, μs = load();
@info "||images||=$N"

[ Info: Loading


  0.063419 seconds (30.72 k allocations: 11.784 MiB, 47.50% compilation time)


[ Info: ||images||=1281167


# Rudimentary Auto Encoder

In [3]:
encoder = Chain(
	Conv((3,3), 3 => 32, stride=2, pad=1),  # 32×32×32
	BatchNorm(32),
	relu,

	Conv((3,3), 32 => 64, stride=2, pad=1), # 16×16×64
	BatchNorm(64),
	relu,

	Conv((3,3), 64 => 128, stride=2, pad=1), # 8×8×128
	BatchNorm(128),
	relu,

	Conv((3,3), 128 => 256, stride=2, pad=1), # 4×4×256
	BatchNorm(256),
	relu,

	# Conv((3,3), 256 => 512, stride=2, pad=1), # 2×2×512
	# BatchNorm(512),
	# relu
)

decoder = Chain(
	# ConvTranspose((3,3), 512 => 256, stride=2, pad=1, outpad=1),
	# BatchNorm(256),
	# relu,

	ConvTranspose((3,3), 256 => 128, stride=2, pad=1, outpad=1),
	BatchNorm(128),
	relu,

	ConvTranspose((3,3), 128 => 64, stride=2, pad=1, outpad=1),
	BatchNorm(64),
	relu,

	ConvTranspose((3,3), 64 => 32, stride=2, pad=1, outpad=1),
	BatchNorm(32),
	relu,

	ConvTranspose((3,3), 32 => 3, stride=2, pad=1, outpad=1),
	sigmoid               # outputs in [0,1]
)

model=Chain(encoder, decoder);

# Training

In [ ]:
const LOG_DIR = "/data/tensorboard/ae_lux"
ispath(LOG_DIR) && rm(LOG_DIR, force=true, recursive=true)

const CHKPT_DIR = "build/checkpoint"
ispath(CHKPT_DIR) && rm(CHKPT_DIR, force=true, recursive=true)
mkpath(CHKPT_DIR)

tblog = TBLogger(LOG_DIR)

@info "Training"
@time with_logger(tblog) do
    dev = gpu_device()
    rng = Random.default_rng()
    ps, st = Lux.setup(rng, model)
    
    ps = ps |> dev
    st = st |> dev
    
    opt = Optimisers.Adam(1f-3)
    opt_state = Optimisers.setup(opt, ps)
    
    batchsize = 1024
    nepochs = 10
	xb_gpu = CUDA.zeros(Float32, 64, 64, 3, batchsize)

    for epoch ∈ 1:nepochs
        for k ∈ randperm(length(Xs))
            Xk = Xs[k]                  # (64,64,3,Nk)
            Nk = size(Xk, 4)
    
			starts = collect(1:batchsize:Nk)
			starts = starts[randperm(length(starts))]

			for (loop, i) ∈ enumerate(starts)
				j = min(i + batchsize - 1, Nk)
				nb = j - i + 1
                nb < batchsize &&  continue
				copyto!(xb_gpu, Xk[:, :, :, i:j])

                (loss, st_new), back = Zygote.pullback(ps) do ps_
                    ŷ, s = Lux.apply(model, xb_gpu, ps_, st)
                    loss = mean((ŷ .- xb_gpu).^2)
                    return loss, s
                end
    
                ∇l = back((1f0, nothing))[1]
                opt_state, ps = Optimisers.update(opt_state, ps, ∇l)
                st = st_new
    
                # progress + loss
                @info "training" loss=loss
            end
        end
    
        @info "training" epoch="epoch=$epoch done"
        let
            cpu = cpu_device()
            ps_cpu = ps |> cpu
            st_cpu = st |> cpu
            @save "$CHKPT_DIR/model_$(epoch).jld2" model=model parameters=ps_cpu state=st_cpu
        end
    end
end;

[ Info: Training


# Quick Check

In [ ]:
model = nothing
parameters = nothing
state = nothing
@load "$CHKPT_DIR/model_10.jld2" model parameters state

In [ ]:
meta = matread("/data/imagenet/2012/ILSVRC2012_devkit_t12/data/meta.mat")
synsets = meta["synsets"];

# synsets["WNID"]
# synsets["wordnet_height"]
# synsets["num_children"]
# synsets["num_train_images"]
# synsets["ILSVRC2012_ID"]
# synsets["words"]
# synsets["gloss"]
# synsets["children"]

In [ ]:
function predict(img)
    x = reshape(img, 64, 64, 3, 1)
    st_eval = Lux.testmode(state)
    ŷ, st_new = Lux.apply(model, x, parameters, st_eval)
    return Array(ŷ)[:, :, :, 1]
end

shard = 5
index = 1001
test_img = Xs[shard][:,:,:,index]
display(synsets["gloss"][ys[shard][index]])

fig = Figure(size=(400,400))
ax1 = Axis(fig[1, 1])
ax2 = Axis(fig[2, 1])
image!(ax1, Utils.makie_image(test_img))
image!(ax2, Utils.makie_image(predict(test_img)))
hidedecorations!(ax1)
hidespines!(ax1)
hidedecorations!(ax2)
hidespines!(ax2)
fig

Clearly 4x4x256 latent dimension starts to lose data for an input image of 64x64x3.

In [ ]:
ratio = (4 * 4 * 256) / (64 * 64 * 3)

Latent representation is about 1/3 of the original size

## IO

In [ ]:
IO = 64*64*3*N*4*10/1e9
@info "IO: $IO GB"
@info "RATE: $(IO/443) GB/s"

* Figure PCIe 5 ~ 15GB/s
* NVMe 5 ~ near the theoreical PCIe 5 max.